<a href="https://colab.research.google.com/github/MattManson/lastfm-music-pipeline/blob/main/lastfm_medallion_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pyspark

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
  .appName("lastfm-pipeline") \
  .getOrCreate()

print(spark.version)


3.5.8


In [6]:
data = [
    ("PinkPantheress", "electronic", 100),
    ("Harry Styles", "pop", 95),
    ("Central Cee", "hip-hop", 88),
    ("Wet Leg", "indie", 76),
    ("Slowthai", "hip-hop", 71)
]

columns = ["artist_name", "genre", "popularity"]
df = spark.createDataFrame(data, columns)

In [7]:
df.show()

+--------------+----------+----------+
|   artist_name|     genre|popularity|
+--------------+----------+----------+
|PinkPantheress|electronic|       100|
|  Harry Styles|       pop|        95|
|   Central Cee|   hip-hop|        88|
|       Wet Leg|     indie|        76|
|      Slowthai|   hip-hop|        71|
+--------------+----------+----------+



In [8]:
from pyspark.sql.functions import col

filtered_df = df.filter(col("popularity")>80)
filtered_df.show()

+--------------+----------+----------+
|   artist_name|     genre|popularity|
+--------------+----------+----------+
|PinkPantheress|electronic|       100|
|  Harry Styles|       pop|        95|
|   Central Cee|   hip-hop|        88|
+--------------+----------+----------+



In [9]:
from pyspark.sql.functions import avg, count

genre_summary = df.groupBy(col("genre")) \
  .agg(
      count(col("artist_name")).alias("artist_count"),
      avg(col("popularity")).alias("avg_popularity")
  )

genre_summary.show()

+----------+------------+--------------+
|     genre|artist_count|avg_popularity|
+----------+------------+--------------+
|       pop|           1|          95.0|
|electronic|           1|         100.0|
|   hip-hop|           2|          79.5|
|     indie|           1|          76.0|
+----------+------------+--------------+



In [10]:
selected_df = df.select(
    col("artist_name"),
    col("genre"),
    col("popularity").alias("popularity_score")
)

selected_df.show()

+--------------+----------+----------------+
|   artist_name|     genre|popularity_score|
+--------------+----------+----------------+
|PinkPantheress|electronic|             100|
|  Harry Styles|       pop|              95|
|   Central Cee|   hip-hop|              88|
|       Wet Leg|     indie|              76|
|      Slowthai|   hip-hop|              71|
+--------------+----------+----------------+



In [11]:
from IPython.core.display import json
import requests
from google.colab import userdata

API_KEY = userdata.get('LASTFM_API_KEY')

BASE_URL = "http://ws.audioscrobbler.com/2.0/"

def get_top_tracks(country, limit=50):
    """
    Fetch top tracks for a given country from Last.fm
    country: country name as a string e.g. 'united kingdom'
    limit: how many tracks to return, max 50 per page
    """
    url = f"http://ws.audioscrobbler.com/2.0/?method=geo.gettoptracks&country={country}&limit={limit}&api_key={API_KEY}&format=json"

    response = requests.get(url)
    print(response.status_code)
    data = response.json()
    return data

result = get_top_tracks("united kingdom", 5)
print(result)

200
{'tracks': {'track': [{'name': 'Stateside + Zara Larsson', 'duration': '176', 'listeners': '34628', 'mbid': 'ffbf7862-2476-4164-ac32-f5904ccefe0f', 'url': 'https://www.last.fm/music/PinkPantheress/_/Stateside+%252B+Zara+Larsson', 'streamable': {'#text': '0', 'fulltrack': '0'}, 'artist': {'name': 'PinkPantheress', 'mbid': '7441014f-f8f5-494f-81db-ff166fbc078d', 'url': 'https://www.last.fm/music/PinkPantheress'}, 'image': [{'#text': 'https://lastfm.freetls.fastly.net/i/u/34s/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'small'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/64s/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'medium'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/174s/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'large'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'extralarge'}], '@attr': {'rank': '0'}}, {'name': 'American Girls', 'duration': '213', 'listeners': '23489', 'mbid': '2c85fe70-3c0e-4b4

In [12]:
import json
from datetime import datetime

def store_bronze(raw_data, source):
    """
    Convert raw API response to a PySpark DataFrame
    representing the bronze layer - raw data with ingestion timestamp

    raw_data: the raw dictionary returned from the API
    source: a label identifying where the data came from
    """

    # Get current timestamp to record when we ingested this data
    ingested_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Convert the entire raw response to a JSON string
    # We store it as one big string - that's intentional for bronze
    raw_json_string = json.dumps(raw_data)

    # Create a single row DataFrame with the raw data and metadata
    bronze_data = [(source, ingested_at, raw_json_string)]
    bronze_columns = ["source", "ingested_at", "raw_json"]

    bronze_df = spark.createDataFrame(bronze_data, bronze_columns)
    return bronze_df

# Test it
bronze_df = store_bronze(result, "lastfm_geo_toptracks_uk")
bronze_df.show(truncate=50)

+-----------------------+-------------------+--------------------------------------------------+
|                 source|        ingested_at|                                          raw_json|
+-----------------------+-------------------+--------------------------------------------------+
|lastfm_geo_toptracks_uk|2026-03-16 16:12:32|{"tracks": {"track": [{"name": "Stateside + Zar...|
+-----------------------+-------------------+--------------------------------------------------+



In [13]:
!pip install delta-spark==3.3.0

In [14]:
import pyspark
print(pyspark.__version__)

3.5.8


In [15]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

spark = configure_spark_with_delta_pip(
    SparkSession.builder \
        .appName("lastfm-pipeline") \
        .config("spark.sql.extensions",
                "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .master("local[*]")
).getOrCreate()

print(spark.version)

3.5.8


In [16]:
# Define our base path and layer paths
BASE_PATH = "/tmp/lastfm_pipeline"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

print(f"Bronze: {BRONZE_PATH}")
print(f"Silver: {SILVER_PATH}")
print(f"Gold: {GOLD_PATH}")

Bronze: /tmp/lastfm_pipeline/bronze
Silver: /tmp/lastfm_pipeline/silver
Gold: /tmp/lastfm_pipeline/gold


In [17]:
from datetime import datetime
import json

def write_bronze(raw_data, source):
    """
    Write raw API response to the bronze Delta table.
    Bronze stores data exactly as received with a timestamp.
    We append each run rather than overwrite - building up history.
    """
    ingested_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    raw_json_string = json.dumps(raw_data)

    bronze_data = [(source, ingested_at, raw_json_string)]
    bronze_columns = ["source", "ingested_at", "raw_json"]

    bronze_df = spark.createDataFrame(bronze_data, bronze_columns)

    # Write to Delta format - append means we add to existing data
    # rather than overwriting it each run
    bronze_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{BRONZE_PATH}/raw_api_responses")

    print(f"Bronze write complete: {source} at {ingested_at}")
    return bronze_df

# Test it
write_bronze(result, "lastfm_geo_toptracks_uk")

Bronze write complete: lastfm_geo_toptracks_uk at 2026-03-16 16:12:38


DataFrame[source: string, ingested_at: string, raw_json: string]

In [18]:
import os

# List everything Delta created at our bronze path
for root, dirs, files in os.walk(f"{BRONZE_PATH}/raw_api_responses"):
    level = root.replace(f"{BRONZE_PATH}/raw_api_responses", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

raw_api_responses/
  .part-00000-c10cb482-c929-42a6-a314-80b97ed15739-c000.snappy.parquet.crc
  part-00000-c10cb482-c929-42a6-a314-80b97ed15739-c000.snappy.parquet
  part-00001-0d129d24-67a6-4d9a-aa07-d82c644205b7-c000.snappy.parquet
  .part-00001-0d129d24-67a6-4d9a-aa07-d82c644205b7-c000.snappy.parquet.crc
  _delta_log/
    00000000000000000000.json
    .00000000000000000000.crc.crc
    00000000000000000000.crc
    .00000000000000000000.json.crc
    _commits/
